In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

pd.set_option('display.float_format', '{:.6f}'.format)
%matplotlib inline


In [8]:
IN_PATH = 'CLEANED DATA/REMX_prices_sentiment_combined.xlsx'

df = pd.read_excel(IN_PATH, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
df = df.set_index('Date')
df['r'] = np.log(df['Close'] / df['Close'].shift(1))
r  = df['r'].dropna()
r2 = (r ** 2)


# Robustness checks

### Why a Student-$t$ innovation specification?

The baseline EGARCH(1,1)-X was estimated under the assumption of Gaussian innovations, $z_t \sim \mathcal{N}(0,1)$. There are two reasons to revisit this choice.

**1. The empirical evidence points to fat tails.** The exploratory analysis of REMX log returns reported an excess kurtosis of $\approx 3.7$ (almost three times the Gaussian benchmark of zero). This is a textbook signal that the unconditional return distribution has heavier tails than the normal — extreme daily moves occur more often than a Gaussian model would predict. A volatility model whose innovations are forced to be Gaussian is therefore forced to inflate $\sigma_t^2$ to "explain" tail observations that are really driven by the shape of $z_t$, contaminating the variance dynamics and the sentiment coefficients.

**2. Quasi-MLE robustness is asymptotic, not free.** Under correct mean-and-variance specification, the Gaussian MLE remains consistent for the GARCH/EGARCH parameters even when the innovations are non-Gaussian — this is the Quasi-MLE result of Bollerslev and Wooldridge (1992) — but the standard errors and $t$-statistics suffer a finite-sample efficiency loss that can be substantial when the true tails are heavy. In our Gaussian EGARCH-X estimates, $\gamma_{\text{tone}}$ and $\gamma_{\text{news}}$ landed at $p \approx 0.34$ and $p \approx 0.36$. If the true innovation distribution is leptokurtic, the conservative Gaussian SEs may be hiding genuine sentiment effects: the point estimates could still be informative, but they are tested against an artificially inflated yardstick.

**The remedy: Student-$t$ innovations.** We replace $z_t \sim \mathcal{N}(0,1)$ with the standardized Student-$t$ distribution with $\nu$ degrees of freedom, $z_t \sim t_\nu^{\star}$, $\mathbb{E}[z_t] = 0$, $\mathrm{Var}(z_t) = 1$. The parameter $\nu$ is estimated jointly with the rest of the model by Maximum Likelihood; small values of $\nu$ correspond to heavy tails, and the Gaussian model is nested as the limit $\nu \to \infty$. This re-specification both (i) improves the fit to the tails of the standardized residual distribution and (ii) re-prices the standard errors of every coefficient — including the sentiment loadings $\gamma_{\text{tone}}$ and $\gamma_{\text{news}}$ — under the correct likelihood.

### The likelihood-ratio test of fat tails

To assess whether the Gaussian assumption can be statistically rejected against the Student-$t$ alternative, we use the **likelihood-ratio (LR) test**. Because the two models are nested ($\nu \to \infty$ recovers the Gaussian), the LR statistic

$$\mathrm{LR} \;=\; 2\,\bigl[\,\ell_{\text{Student-}t}(\hat\theta_t) \;-\; \ell_{\text{Gaussian}}(\hat\theta_g)\,\bigr]$$

is asymptotically distributed as $\chi^2(1)$ under the null hypothesis $H_0:\;\nu = \infty$ (Gaussian innovations). Rejection at the 5% (resp. 1%) level requires $\mathrm{LR} > 3.841$ (resp. $> 6.635$).

A formal caveat applies: $\nu = \infty$ is a boundary point of the parameter space, so the strict asymptotic distribution of LR is a 50:50 mixture of $\chi^2(0)$ and $\chi^2(1)$ (Self & Liang, 1987). In practice this halves the $p$-value, making the test slightly more powerful against the Gaussian null. For our purposes the standard $\chi^2(1)$ reference is reported as a conservative benchmark; for daily financial returns it is essentially never tight.

**Interpretation rule:**

- If $\mathrm{LR}$ is large and $\hat\nu$ is small (e.g. $\hat\nu < 10$): the Student-$t$ specification is preferred — fat tails are statistically significant, and the sentiment coefficients should be re-read with the new $p$-values rather than the Gaussian ones.
- If $\mathrm{LR}$ is small and $\hat\nu$ is large (e.g. $\hat\nu > 30$): the data are essentially Gaussian and the Student-$t$ refinement does not change the conclusions.
- AIC and BIC are reported alongside to confirm the verdict from an information-criterion standpoint: a lower AIC/BIC under Student-$t$ is consistent with the LR rejection of the Gaussian null.


In [9]:
%%capture cap_egarch_x_studentt
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm, chi2
from scipy.special import gammaln                          # *** NEW *** (Student-t density)
from statsmodels.tools.numdiff import approx_hess

# ── 1. Aligned dataset: r_t (in %), Tone_{t-1}, log_article_count_{t-1} ──────────
r_pct    = df['r'] * 100.0
tone_lag = df['tone_mean'].shift(1)
vol_lag  = df['art_growth'].shift(1)

aligned = pd.concat(
    [r_pct, tone_lag, vol_lag],
    axis=1, keys=['r', 'tone_lag', 'vol_lag']
).dropna()

r    = aligned['r'].values
tone = aligned['tone_lag'].values
vol  = aligned['vol_lag'].values
T    = len(r)

print(f'Sample after alignment: T = {T} obs '
      f'({aligned.index.min().date()} -> {aligned.index.max().date()})')

# ── 2. EGARCH(1,1)-X recursion with Student-t innovations ─────────────────
# *** NEW *** PARAM_NAMES gains 'nu'
PARAM_NAMES = ['mu', 'omega', 'alpha', 'xi', 'beta', 'gamma_tone', 'gamma_vol', 'nu']
CLAMP = 30.0

def expected_abs_z_t(nu):
    """*** NEW *** E|z| for the standardized Student-t with degrees of freedom nu."""
    return (2.0 * np.sqrt((nu - 2.0) / np.pi)
            * np.exp(gammaln((nu + 1.0) / 2.0) - gammaln(nu / 2.0))
            / (nu - 1.0))

def per_obs_ll(params, r, tone, vol):
    mu, omega, alpha, xi, beta, g1, g2, nu = params       # *** NEW *** 8 params
    n = len(r)
    eps = r - mu
    log_sig2 = np.empty(n)
    log_sig2[0] = np.log(max(np.var(eps), 1e-8))

    e_abs_z = expected_abs_z_t(nu)                        # *** NEW *** depends on nu

    for t in range(1, n):
        z_prev = eps[t-1] / np.exp(0.5 * log_sig2[t-1])
        ls = (omega
              + alpha * (np.abs(z_prev) - e_abs_z)
              + xi    * z_prev
              + beta  * log_sig2[t-1]
              + g1    * tone[t]
              + g2    * vol[t])
        log_sig2[t] = np.clip(ls, -CLAMP, CLAMP)

    sig2 = np.exp(log_sig2)
    z = eps / np.sqrt(sig2)

    # *** NEW *** Standardized Student-t log-density (replaces Gaussian density)
    const = (gammaln((nu + 1.0) / 2.0) - gammaln(nu / 2.0)
             - 0.5 * np.log(np.pi * (nu - 2.0)))
    ll_t = (const
            - 0.5 * (nu + 1.0) * np.log(1.0 + z**2 / (nu - 2.0))
            - 0.5 * log_sig2)
    return ll_t, log_sig2

def neg_ll(params, r, tone, vol):
    ll_t, _ = per_obs_ll(params, r, tone, vol)
    return -ll_t.sum()

# ── 3. Maximum-likelihood estimation ──────────────────────────────────────
# *** NEW *** Initial value for nu (8 is a common starting point for daily returns)
x0 = np.array([-0.0191, 0.0487, 0.1790, -0.0244, 0.9806, 0.0074, -0.0062, 8.0])

# *** NEW *** Bounds (especially nu > 2)
bounds = [(None,  None),       # mu
          (None,  None),       # omega
          (None,  None),       # alpha
          (None,  None),       # xi
          (-0.999, 0.999),     # beta  (stationarity)
          (None,  None),       # gamma_tone
          (None,  None),       # gamma_vol
          (2.05,  100.0)]      # nu    (finite variance requires nu > 2)

opt = minimize(neg_ll, x0, args=(r, tone, vol),
               method='L-BFGS-B', bounds=bounds,
               options={'disp': False, 'maxiter': 5000})
if not opt.success:
    print(f'WARNING: optimizer message: {opt.message}')

theta = opt.x
ll    = -opt.fun

# ── 4. Robust (sandwich) standard errors ──────────────────────────────────
H      = approx_hess(theta, neg_ll, args=(r, tone, vol))
V_hess = np.linalg.inv(H)

eps_fd = 1e-5
G = np.empty((T, len(theta)))
for i in range(len(theta)):
    p_up = theta.copy(); p_up[i] += eps_fd
    p_dn = theta.copy(); p_dn[i] -= eps_fd
    ll_up, _ = per_obs_ll(p_up, r, tone, vol)
    ll_dn, _ = per_obs_ll(p_dn, r, tone, vol)
    G[:, i] = (ll_up - ll_dn) / (2 * eps_fd)

J        = G.T @ G
V_robust = V_hess @ J @ V_hess
se       = np.sqrt(np.diag(V_robust))
t_ratios = theta / se
p_values = 2.0 * (1.0 - norm.cdf(np.abs(t_ratios)))

# ── 5. Information criteria ───────────────────────────────────────────────
k   = len(theta)
aic = -2.0 * ll + 2.0 * k
bic = -2.0 * ll + k * np.log(T)

# ── 6. Report (same format) ───────────────────────────────────────────────
tbl = pd.DataFrame({
    'estimate': theta,
    'std_err':  se,
    't_ratio':  t_ratios,
    'p_value':  p_values,
}, index=PARAM_NAMES)
tbl['significant_10pct'] = tbl['t_ratio'].abs() > 1.645
tbl['significant_5pct']  = tbl['t_ratio'].abs() > 1.96
tbl['significant_1pct']  = tbl['t_ratio'].abs() > 2.576

print()
print('Per-parameter significance (|t| > 1.645 -> 10%, |t| > 1.96 -> 5%, |t| > 2.576 -> 1%):')
print(tbl.to_string(float_format=lambda x: f'{x: .4f}'))
print('Note: the t-test on nu against zero is not economically meaningful (nu > 2 by construction).')
print('      Use the LR test below to assess fat-tailedness vs the Gaussian specification.')
print()
print(f'Log-likelihood : {ll: .4f}')
print(f'AIC            : {aic: .4f}')
print(f'BIC            : {bic: .4f}')
print()
beta_hat = theta[4]
nu_hat   = theta[7]
print(f'beta = {beta_hat:.4f}   '
      f'({"stationary (|beta|<1)" if abs(beta_hat) < 1 else "non-stationary (|beta|>=1)"})')
half_life = np.log(0.5) / np.log(abs(beta_hat)) if 0 < abs(beta_hat) < 1 else float('inf')
print(f'Half-life of a log-variance shock: {half_life:.1f} trading days')
print(f'Estimated degrees of freedom nu = {nu_hat:.2f}   '
      f'(smaller = heavier tails; Gaussian as nu -> inf)')

# *** NEW *** LR test: Student-t (this model) vs Gaussian EGARCH-X
# Load Gaussian EGARCH-X log-likelihood from notebook 04 (with fallback)
import json
from pathlib import Path
_ll_path = Path('REPORTS') / 'egarch_x_normal_ll.json'
if _ll_path.exists():
    with _ll_path.open() as fh:
        LL_GAUSS_EGARCH_X = json.load(fh)['ll_gauss_egarch_x']
else:
    LL_GAUSS_EGARCH_X = -5972.9063   # fallback if 04 not yet run
LR_fat = 2.0 * (ll - LL_GAUSS_EGARCH_X)
p_fat  = 1.0 - chi2.cdf(LR_fat, df=1)
print()
print('LR test of fat tails (Student-t vs Gaussian EGARCH-X):')
print(f'  LR = 2(LL_t - LL_normal) = 2({ll:.4f} - {LL_GAUSS_EGARCH_X:.4f}) = {LR_fat:.4f}')
print(f'  Reference chi^2(1):  5% = 3.841,  1% = 6.635')
print(f'  p-value           : {p_fat:.4f}   '
      f'({"REJECT Gaussian" if LR_fat > 3.841 else "Cannot reject Gaussian"} at 5%)')

print()
print('Verdict:')
for name, row in tbl.iterrows():
    if row['significant_1pct']:
        tag = 'SIGNIFICANT at 1% (and 5%, 10%)'
    elif row['significant_5pct']:
        tag = 'SIGNIFICANT at 5% (and 10%) only'
    elif row['significant_10pct']:
        tag = 'SIGNIFICANT at 10% only'
    else:
        tag = 'not significant'
    print(f'  {name:<12}  t = {row["t_ratio"]:+.2f}   p = {row["p_value"]:.4f}   -> {tag}')


# Generate Report

In [10]:
from pathlib import Path

REPORTS_DIR = Path("REPORTS")
REPORTS_DIR.mkdir(exist_ok=True)
(REPORTS_DIR / "_capture_egarch_x_studentt_art_growth.txt").write_text(cap_egarch_x_studentt.stdout)

print(cap_egarch_x_studentt.stdout)
print(f"Persisted capture -> REPORTS/_capture_egarch_x_studentt_art_growth.txt")


Sample after alignment: T = 2765 obs (2015-04-02 -> 2026-03-31)

Per-parameter significance (|t| > 1.645 -> 10%, |t| > 1.96 -> 5%, |t| > 2.576 -> 1%):
            estimate  std_err   t_ratio  p_value  significant_10pct  significant_5pct  significant_1pct
mu           -0.0155   0.0367   -0.4228   0.6725              False             False             False
omega         0.0316   0.0099    3.2025   0.0014               True              True              True
alpha         0.1541   0.0243    6.3448   0.0000               True              True              True
xi           -0.0148   0.0108   -1.3769   0.1685              False             False             False
beta          0.9785   0.0063  154.3819   0.0000               True              True              True
gamma_tone    0.0104   0.0049    2.1231   0.0337               True              True             False
gamma_vol    -0.0172   0.0098   -1.7480   0.0805               True             False             False
nu           10.8